# 3d

In [1]:
import ansys.aedt.core
import os
import tempfile
import time
AEDT_VERSION = "2024.1"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.
from ansys.aedt.core import Desktop


c:\Users\HAN_NDESKTOP\.ansys_python_venvs\pyAEDT_conda\Lib\site-packages\ansys\aedt\core\modeler\schematic.py:39: UserWarning: EMIT API is only available for Python 3.8-3.12.
  warnings.warn("EMIT API is only available for Python 3.8-3.12.")


In [3]:
get_aedt_processes_detailed()

🔍 실행 중인 AEDT 프로세스 검색...
❌ AEDT 프로세스가 없습니다.


[]

In [ ]:

new_desktop=try_connect_with_ports([17004])

In [2]:
import ansys.aedt.core
from ansys.aedt.core import Desktop
import subprocess
import psutil
import time
import os
import re

def get_aedt_processes_detailed():
    """
    실행 중인 AEDT 프로세스를 상세히 확인합니다.
    
    Returns:
    --------
    list : AEDT 프로세스 정보 리스트
    """
    print("🔍 실행 중인 AEDT 프로세스 검색...")
    
    aedt_processes = []
    try:
        for proc in psutil.process_iter(['pid', 'name', 'cmdline', 'create_time']):
            try:
                pinfo = proc.info
                process_name = pinfo['name'] if pinfo['name'] else ""
                
                # AEDT 관련 프로세스 필터링
                if any(keyword in process_name.lower() for keyword in ['ansysedt', 'aedt']):
                    # 포트 정보 추출 시도
                    ports = []
                    try:
                        connections = proc.connections()
                        for conn in connections:
                            if conn.status == 'LISTEN':
                                ports.append(conn.laddr.port)
                    except (psutil.AccessDenied, psutil.NoSuchProcess):
                        pass
                    
                    aedt_processes.append({
                        'pid': pinfo['pid'],
                        'name': process_name,
                        'cmdline': pinfo['cmdline'] if pinfo['cmdline'] else [],
                        'create_time': time.ctime(pinfo['create_time']),
                        'ports': ports
                    })
                    
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
    
        if aedt_processes:
            print(f"✅ {len(aedt_processes)}개의 AEDT 프로세스 발견:")
            for i, proc in enumerate(aedt_processes):
                print(f"\n📋 프로세스 {i+1}:")
                print(f"   PID: {proc['pid']}")
                print(f"   이름: {proc['name']}")
                print(f"   생성시간: {proc['create_time']}")
                if proc['ports']:
                    print(f"   열린 포트: {proc['ports']}")
                else:
                    print(f"   열린 포트: 없음")
        else:
            print("❌ AEDT 프로세스가 없습니다.")
            
        return aedt_processes
        
    except Exception as e:
        print(f"❌ 프로세스 검색 중 오류: {e}")
        return []

def try_connect_to_existing_desktop():
    """
    기존 AEDT Desktop에 연결을 시도합니다.
    
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    print("🔗 기존 AEDT Desktop 연결 시도...")
    
    try:
        # 방법 1: new_desktop_session=False로 기존 세션에 연결
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=False,
            non_graphical=NG_MODE
        )
        print("✅ 기존 AEDT Desktop에 성공적으로 연결되었습니다!")
        return desktop
        
    except Exception as e:
        print(f"❌ 기존 Desktop 연결 실패: {e}")
        return None

def try_connect_with_ports(port_list):
    """
    특정 포트들을 시도해서 AEDT에 연결합니다.
    
    Parameters:
    -----------
    port_list : list
        시도할 포트 번호 리스트
        
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    AEDT_VERSION='251'
    NG_MODE=False
    for port in port_list:
        try:
            print(f"🔗 포트 {port}로 연결 시도...")
            desktop = Desktop(
                specified_version=AEDT_VERSION,
                new_desktop_session=False,
                port=port,
                non_graphical=NG_MODE
            )
            print(f"✅ 포트 {port}로 성공적으로 연결되었습니다!")
            return desktop
        except Exception as e:
            print(f"❌ 포트 {port} 연결 실패: {e}")
            continue
    
    return None

def get_desktop_connection():
    """
    다양한 방법으로 AEDT Desktop 연결을 시도합니다.
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("=" * 60)
    print("🎯 AEDT Desktop 연결 시도")
    print("=" * 60)
    
    # 1. 기존 Desktop 연결 시도
    desktop = try_connect_to_existing_desktop()
    if desktop:
        return desktop
    
    # 2. 프로세스에서 포트 찾아서 연결 시도
    processes = get_aedt_processes_detailed()
    all_ports = []
    
    for proc in processes:
        all_ports.extend(proc['ports'])
    
    if all_ports:
        desktop = try_connect_with_ports(all_ports)
        if desktop:
            return desktop
    
    # 3. 일반적인 AEDT 포트들 시도
    common_ports = [56800, 56801, 56802, 56803, 56804, 56805]
    print("\n🔍 일반적인 AEDT 포트들 시도...")
    desktop = try_connect_with_ports(common_ports)
    if desktop:
        return desktop
    
    # 4. 새로운 Desktop 세션 생성
    print("\n🆕 새로운 AEDT Desktop 세션을 생성합니다...")
    try:
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
        print("✅ 새로운 AEDT Desktop이 생성되었습니다!")
        return desktop
    except Exception as e:
        print(f"❌ 새 Desktop 생성 실패: {e}")
        return None

def check_current_desktop_status(desktop):
    """
    현재 Desktop의 상태를 확인합니다.
    
    Parameters:
    -----------
    desktop : Desktop
        확인할 Desktop 객체
    """
    if not desktop:
        print("❌ Desktop 객체가 없습니다.")
        return
    
    try:
        print("\n" + "=" * 40)
        print("📊 현재 Desktop 상태:")
        print("=" * 40)
        
        # 기본 정보
        print(f"AEDT 버전: {desktop.aedt_version_id}")
        print(f"프로세스 ID: {desktop.aedt_process_id}")
        
        # 프로젝트 정보
        try:
            projects = desktop.project_list()
            print(f"\n📁 열린 프로젝트 ({len(projects)}개):")
            for i, proj_name in enumerate(projects, 1):
                print(f"  {i}. {proj_name}")
            
            # 활성 프로젝트
            active_proj = desktop.active_project()
            if active_proj:
                proj_name = active_proj.GetName()
                print(f"\n🎯 활성 프로젝트: {proj_name}")
                
                # 디자인 목록
                try:
                    design_list = active_proj.GetTopDesignList()
                    print(f"📐 디자인 ({len(design_list)}개):")
                    for i, design in enumerate(design_list, 1):
                        print(f"  {i}. {design}")
                        
                    # 활성 디자인
                    active_design = desktop.active_design()
                    if active_design:
                        print(f"🎯 활성 디자인: {active_design.GetName()}")
                        print(f"   디자인 타입: {active_design.GetDesignType()}")
                except:
                    print("디자인 정보 가져오기 실패")
            else:
                print("🎯 활성 프로젝트: 없음")
                
        except Exception as e:
            print(f"프로젝트 정보 가져오기 실패: {e}")
            
    except Exception as e:
        print(f"❌ Desktop 상태 확인 중 오류: {e}")

def smart_aedt_connector():
    """
    스마트 AEDT 연결 함수 - 사용자 친화적 인터페이스
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("🚀 스마트 AEDT 연결기를 시작합니다...")
    
    # Desktop 연결 시도
    desktop = get_desktop_connection()
    
    if desktop:
        # 연결 성공 시 상태 확인
        check_current_desktop_status(desktop)
        
        print("\n" + "=" * 60)
        print("🎉 AEDT Desktop 연결이 완료되었습니다!")
        print("💡 다음과 같이 사용할 수 있습니다:")
        print("=" * 60)
        print("# 프로젝트 열기:")
        print("# project = desktop.open_project(r'C:\\path\\to\\your\\project.aedt')")
        print("#")
        print("# Maxwell 객체 생성:")
        print("# m2d = ansys.aedt.core.Maxwell2d(project=desktop, new_desktop=False)")
        print("# m3d = ansys.aedt.core.Maxwell3d(project=desktop, new_desktop=False)")
        print("=" * 60)
        
        return desktop
    else:
        print("❌ AEDT Desktop 연결에 실패했습니다.")
        print("\n🔍 문제 해결 방법:")
        print("1. Ansys AEDT가 설치되어 있는지 확인")
        print("2. AEDT 라이선스가 사용 가능한지 확인")
        print("3. 수동으로 AEDT를 실행한 후 다시 시도")
        return None

# 간단한 사용 함수들
def quick_connect():
    """빠른 연결 - 기존 세션 우선"""
    return try_connect_to_existing_desktop()

def force_new_session():
    """강제로 새 세션 생성"""
    try:
        return Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
    except Exception as e:
        print(f"새 세션 생성 실패: {e}")
        return None


In [ ]:
desktop = Desktop(specified_version=AEDT_VERSION, new_desktop_session=False)


In [ ]:
desktop = Desktop(specified_version=AEDT_VERSION, new_desktop_session=False,)


In [ ]:
# AEDT 실행 연결 (GUI 띄우지 않으려면 non_graphical=True)
desktop = Desktop(specified_version=AEDT_VERSION, new_desktop_session=False)

# # 프로젝트 열기
# proj = desktop.open_project(r"C:\your_path\your_project.aedt")

# # 디자인 이름 목록 추출
# design_names = proj.design_list
# print("Designs in the project:", design_names)


In [ ]:
'D:\JHC\2023_KAU\2025-05-15_\KAU_model1.aedt'

In [4]:
# project_name = os.path.join("D:\EM_KDH\[1]PJT\[0]BMC_Technical_Support\[1]8p12s", "8p12s.aedt")
# project_name = os.path.join('D:\EM_KDH\[1]PJT\[0]BMC_Technical_Support\[1]8p12s', "8p12s_ANSYSEM_2D.aedt")
# project_name = r"D:\KDH\BMC\WindingSetup_20250514\EMB MOTOR_Simplify_R251.aedt"
# project_name=r'D:\KangDH\deVSimulation\Ansys\BMC\EMB MOTOR_Simplify_R251.aedt'
# project_name=r"D:\KangDH\deVSimulation\Ansys\BMC\EMB MOTOR_Simplify_R251.aedt"
# project_name=r"D:\KangDH\deVSimulation\Ansys\BMC\EMB MOTOR_load_R251.aedt"
# project_name=r"D:/KangDH/deVSimulation/Ansys/BMC/Project1.aedt"
# project_name=r'D:\KangDH\pyAEDT\pyaedt\tests\system\general\example_models\TMaxwell\Motor3D_cyl_gap.aedt'
project_name=r'D:\KangDH\pyAEDT\pyaedt\tests\system\general\example_models\TMaxwell\Transient_StrandedWindings.aedt'
# project_name=r"G:/SIS/25.xx.xx [HMC] 20Deg_1/AFPM_4pitch_Full_Seg_Mag_reference.aedt"

In [ ]:
curdesign=desktop.active_design()

In [ ]:
curdesign

In [5]:
m3d = Maxwell3d(project=project_name,
    # design='Maxwell2DDesign1',
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE,
)

NameError: name 'Maxwell3d' is not defined

In [ ]:
designName=m3d.design_list

In [ ]:
desktop.close_project()

In [ ]:
all_objects = m3d.modeler.object_names
for obj_name in all_objects:
    obj = m3d.modeler[obj_name]



In [ ]:
ob3dlist[1].faces

In [ ]:
materials=m3d.materials.material_keys
excitations=m3d.excitation_names

In [ ]:
boundaryObjs=m3d.boundaries


In [ ]:
bName = []
bproperties = []
for boundaryObj in boundaryObjs:
    bName.append(boundaryObj.name)
    bproperties.append(boundaryObj.properties)


In [ ]:
m3d.close_project(save_project=False)
# m2d.close_desktop()

# Report 

In [ ]:
# report_torque = m2d.post.create_report(
#     expressions="Moving1.Torque",
#     domain="Sweep",
#     variations={"bridge": "All", "din": "All", "Ipeak": "All", "phase_advance": "All"},
#     primary_sweep_variable="Time",
#     plot_type="Rectangular Plot",
#     plot_name="TorqueAllVariations",
# )

### OutputVars

In [ ]:
output_vars = {
    # "Current_A": "InputCurrent(Phase_A)",
    # "Current_B": "InputCurrent(Phase_B)",
    # "Current_C": "InputCurrent(Phase_C)",
    "Flux_A": "FluxLinkage(U)",
    "Flux_B": "FluxLinkage(U1)",
    "Flux_C": "FluxLinkage(V1)",
    # "pos": "(Moving1.Position -InitialPositionMD) *NumPoles/2",
    # "cos0": "cos(pos)",
    # "cos1": "cos(pos-2*PI/3)",
    # "cos2": "cos(pos-4*PI/3)",
    # "sin0": "sin(pos)",
    # "sin1": "sin(pos-2*PI/3)",
    # "sin2": "sin(pos-4*PI/3)",
    # "Flux_d": "2/3*(Flux_A*cos0+Flux_B*cos1+Flux_C*cos2)",
    # "Flux_q": "-2/3*(Flux_A*sin0+Flux_B*sin1+Flux_C*sin2)",
    # "I_d": "2/3*(Current_A*cos0 + Current_B*cos1 + Current_C*cos2)",
    # "I_q": "-2/3*(Current_A*sin0 + Current_B*sin1 + Current_C*sin2)",
    # "Irms": "sqrt(I_d^2+I_q^2)/sqrt(2)",
    # "ArmatureOhmicLoss_DC": "Irms^2*R_phase",
    # "Lad": "L(Phase_A,Phase_A)*cos0 + L(Phase_A,Phase_B)*cos1 + L(Phase_A,Phase_C)*cos2",
    # "Laq": "L(Phase_A,Phase_A)*sin0 + L(Phase_A,Phase_B)*sin1 + L(Phase_A,Phase_C)*sin2",
    # "Lbd": "L(Phase_B,Phase_A)*cos0 + L(Phase_B,Phase_B)*cos1 + L(Phase_B,Phase_C)*cos2",
    # "Lbq": "L(Phase_B,Phase_A)*sin0 + L(Phase_B,Phase_B)*sin1 + L(Phase_B,Phase_C)*sin2",
    # "Lcd": "L(Phase_C,Phase_A)*cos0 + L(Phase_C,Phase_B)*cos1 + L(Phase_C,Phase_C)*cos2",
    # "Lcq": "L(Phase_C,Phase_A)*sin0 + L(Phase_C,Phase_B)*sin1 + L(Phase_C,Phase_C)*sin2",
    # "L_d": "(Lad*cos0 + Lbd*cos1 + Lcd*cos2) * 2/3",
    # "L_q": "(Laq*sin0 + Lbq*sin1 + Lcq*sin2) * 2/3",
    # "OutputPower": "Moving1.Speed*Moving1.Torque",
    # "Ui_A": "InducedVoltage(Phase_A)",
    # "Ui_B": "InducedVoltage(Phase_B)",
    # "Ui_C": "InducedVoltage(Phase_C)",
    # "Ui_d": "2/3*(Ui_A*cos0 + Ui_B*cos1 + Ui_C*cos2)",
    # "Ui_q": "-2/3*(Ui_A*sin0 + Ui_B*sin1 + Ui_C*sin2)",
    # "U_A": "Ui_A+R_Phase*Current_A",
    # "U_B": "Ui_B+R_Phase*Current_B",
    # "U_C": "Ui_C+R_Phase*Current_C",
    # "U_d": "2/3*(U_A*cos0 + U_B*cos1 + U_C*cos2)",
    # "U_q": "-2/3*(U_A*sin0 + U_B*sin1 + U_C*sin2)",
}

In [ ]:
for k, v in output_vars.items():
    m2d.create_output_variable(k, v)

In [ ]:
post_params = {"Moving1.Torque": "FluxPlot"}


##  def Rect Plot

### Initialize definition for postprocessing multiplots


#### default 3phase

In [ ]:
post_params_multiplot = {  # reports
    ("U_A", "U_B", "U_C", "Ui_A", "Ui_B", "Ui_C"): "PhaseVoltages",
    ("CoreLoss", "SolidLoss", "ArmatureOhmicLoss_DC"): "Losses",
    (
        "InputCurrent(Phase_A)",
        "InputCurrent(Phase_B)",
        "InputCurrent(Phase_C)",
    ): "PhaseCurrents",
    (
        "FluxLinkage(Phase_A)",
        "FluxLinkage(Phase_B)",
        "FluxLinkage(Phase_C)",
    ): "PhaseFluxes",
    ("I_d", "I_q"): "Currents_dq",
    ("Flux_d", "Flux_q"): "Fluxes_dq",
    ("Ui_d", "Ui_q"): "InducedVoltages_dq",
    ("U_d", "U_q"): "Voltages_dq",
    (
        "L(Phase_A,Phase_A)",
        "L(Phase_B,Phase_B)",
        "L(Phase_C,Phase_C)",
        "L(Phase_A,Phase_B)",
        "L(Phase_A,Phase_C)",
        "L(Phase_B,Phase_C)",
    ): "PhaseInductances",
    ("L_d", "L_q"): "Inductances_dq",
    ("CoreLoss", "CoreLoss(Stator)", "CoreLoss(Rotor)"): "CoreLosses",
    (
        "EddyCurrentLoss",
        "EddyCurrentLoss(Stator)",
        "EddyCurrentLoss(Rotor)",
    ): "EddyCurrentLosses (Core)",
    ("ExcessLoss", "ExcessLoss(Stator)", "ExcessLoss(Rotor)"): "ExcessLosses (Core)",
    (
        "HysteresisLoss",
        "HysteresisLoss(Stator)",
        "HysteresisLoss(Rotor)",
    ): "HysteresisLosses (Core)",
    (
        "SolidLoss",
        "SolidLoss(IPM1)",
        "SolidLoss(IPM1_1)",
        "SolidLoss(OPM1)",
        "SolidLoss(OPM1_1)",
    ): "SolidLoss",
}

In [ ]:
m2d.create_output_variable(
    "SumPhiU",
    "FluxLinkage(U1)+FluxLinkage(U2)",
)
m2d.create_output_variable(  
    "SumPhiW",
    "FluxLinkage(W1)+FluxLinkage(W2)",
)
m2d.create_output_variable(  
    "SumPhiV",
    "FluxLinkage(V1)+FluxLinkage(V2)"
 )

In [ ]:
m2d.create_output_variable(  
    "dVdt",
    "deriv(SumPhiV)/deriv(Time)"
 )
m2d.create_output_variable(  
    "dUdt",
    "deriv(SumPhiU)/deriv(Time)"
 )
m2d.create_output_variable(  
    "dWdt",
    "deriv(SumPhiW)/deriv(Time)"
 )


#### dual Star-Delta

In [ ]:
post_params_multiplot = {  # reports
    (
        "FluxLinkage(U1)",
        "FluxLinkage(V1)",
        "FluxLinkage(W1)",
        "FluxLinkage(U2)",
        "FluxLinkage(V2)",
        "FluxLinkage(W2)",
    ): "Fluxes",
}

In [ ]:
12/7

In [ ]:
import numpy as np
np.sqrt(3)

In [ ]:
all_reported_data = {}
for k, v in post_params_multiplot.items():
    all_reported_data[v] = m2d.post.create_report(
        expressions=list(k),
        setup_sweep_name="",
        domain="Sweep",
        variations=None,
        primary_sweep_variable="Time",
        secondary_sweep_variable=None,
        report_category=None,
        plot_type="Rectangular Plot",
        context=None,
        subdesign_id=None,
        polyline_points=1001,
        plotname=v
    )

# solve

In [ ]:
m2d.analyze()

## plot waveform

In [ ]:
postM2d=   m2d.post

In [ ]:
Fu1=postM2d.get_solution_data(expressions= primary_sweep_variable="Time")

In [ ]:
# ...existing code...

solution_data_dict = {}

for exprs, plot_name in post_params_multiplot.items():
    for expr in exprs:
        try:
            soldata = m2d.post.get_solution_data(
                expressions=expr,
                primary_sweep_variable="Time"
            )
            solution_data_dict[expr] = soldata
        except Exception as e:
            print(f"{expr} 데이터 추출 실패: {e}")

# 결과 예시 출력
for k, v in solution_data_dict.items():
    print(f"{k}: {v}")

# 모든 결과를 한 figure에 플롯
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
for expr, soldata in solution_data_dict.items():
    try:
        plt.plot(soldata.primary_sweep_values, soldata.data_real(), label=expr)
    except Exception as e:
        print(f"{expr} 플롯 실패: {e}")

plt.xlabel("Time[ms]")
plt.ylabel("[Wb]")
# plt.title()
plt.legend()
plt.grid(True)
plt.show()

# ...existing code...

# ...existing code...

In [ ]:
solution_data_dict["FluxLinkage(U1)"].primary_sweep_values

In [ ]:
diff=diffSolData(solution_data_dict)

In [ ]:
# diff["FluxLinkage(U1)"]의 시간 미분 결과 플롯
t, dy_dt = diff["FluxLinkage(U1)"]

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(t, dy_dt, label="d/dt FluxLinkage(U1)")
plt.xlabel("Time")
plt.ylabel("d/dt FluxLinkage(U1)")
plt.title("Time Derivative of FluxLinkage(U1)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def diffSolData(solution_data_dict):
    """
    solution_data_dict의 각 데이터에 대해 시간 미분(dy/dt)을 계산하여 반환합니다.
    반환값은 {expression: (time, 미분값)} 형태의 dict입니다.
    """
    diff_dict = {}
    for expr, soldata in solution_data_dict.items():
        try:
            t = np.array(soldata.primary_sweep_values, dtype=float)
            y = np.array(soldata.data_real(), dtype=float)
            dy_dt = np.gradient(y, t)
            diff_dict[expr] = (t, dy_dt)
        except Exception as e:
            print(f"{expr} 미분 계산 실패: {e}")
    return diff_dict

# 사용 예시:
# diff_data = differentiate_solution_data(solution_data_dict)
# for expr, (t, dy) in diff_data.items():
#     plt.plot(t, dy, label=f"d/dt {expr}")
# plt.legend()
# plt.show()

In [ ]:
m2d.post.available_report_quantities()

In [ ]:
TbSols.enable_pandas_output=True
TbSols.plot(curves=tb.post.available_report_quantities()[3])

In [ ]:
# report_torque = m2d.post.create_report(
#     expressions="Moving1.Torque",
#     domain="Sweep",
#     variations={"bridge": "All", "din": "All", "Ipeak": "All", "phase_advance": "All"},
#     primary_sweep_variable="Time",
#     plot_type="Rectangular Plot",
#     plot_name="TorqueAllVariations",
# )

# Field Plot


In [ ]:
FieldSurfOBj=m2dPost.create_fieldplot_surface(assignment='Stator_Lamination_Primitive',quantity='Mag_B')

In [ ]:
m2dPost.get_model_plotter_geometries(
            generate_mesh=False,
            get_objects_from_aedt=plot_cad_objs,
            plot_as_separate_objects=plot_as_separate_objects,
        )

In [ ]:
ModelPlotObj=m2dPost.plot_field_from_fieldplot(plot_name='Mag_B_4XLK7S')


In [ ]:
ModelPlotterObjByAEDT=ModelPlotObj.pv

In [ ]:
import matplotlib
matplotlib.use('Qt5Agg')

In [ ]:
ModelPlotterObjBddyAEDT

In [ ]:
ModelPlotterObjByAEDT.notebook=False

In [ ]:
ModelPlotterObjByAEDT.off_screen=True

In [ ]:
ModelPlotterObjByAEDT.show()

## Manual ModelPlotter defin

In [ ]:
model

In [ ]:
model.range_max=2
model.range_min=0

In [ ]:
model.fields[0].label='Mag_B'

In [ ]:
field_instance = model.fields[0]  # Select the specific field instance


In [ ]:
pvModel=model.pv

In [ ]:
def callback(point):
    print(f"선택한 노드 좌표: {point}")

pvModel.enable_point_picking(callback=callback, use_mesh=True)

# 인터랙티브 플롯 실행



In [ ]:
pvModel.scalar_bar
pvModel.update_scalar_bar_range([0,2])
pvModel.mapper.GetColorMode()
pvModel.show()

## _pars_aedtplt

In [ ]:
from ansys.aedt.core.visualization.plot.pyvista import _parse_aedtplt 
vertices, faces, scalars, log=_parse_aedtplt("D:\EM_KDH\[1]PJT\[0]BMC_Technical_Support\[1]8p12s\8p12s_Stator_B_MagB4XLK7S.aedtplt")
vertices=vertices[0]
faces=faces[0]
import matplotlib.pyplot as plt

In [ ]:
import matplotlib

In [ ]:
# matplotlib.use("Qt5Agg")

# plt.scatter(vertices[:,0],vertices[:,1])
# plt.interactive(True)
# plt.show()


In [ ]:
import pyvista as pv
mesh = pv.PolyData(vertices, faces)


In [ ]:
plotter = pv.Plotter()
plotter.add_mesh(mesh, show_edges=True, color="lightblue")

## Mesh Plot

In [ ]:
import tkinter as tk
import vtk

pv.set_jupyter_backend('trame')


plotter.add_mesh(mesh, show_edges=True)
# 노드 선택 활성화
def callback(point):
    print(f"선택한 노드 좌표: {point}")

plotter.enable_point_picking(callback=callback, use_mesh=True)

# 인터랙티브 플롯 실행
plotter.show()


# # 노드 선택 활성화
# def cell_callback(cell_id):
#     print(f"선택한 셀 ID: {cell_id}")
# # plotter.enable_point_picking(callback=callback, use_mesh=True)
# plotter.enable_cell_picking(callback=cell_callback, through=False)

# Maxwell 2D GUI 실행

In [ ]:
# Maxwell 2D GUI 창 열기 - 새 프로젝트 생성
from ansys.aedt.core import Maxwell2d
import time

try:
    print("🚀 Maxwell 2D GUI를 실행합니다...")
    
    # AEDT 버전 및 GUI 설정 확인
    AEDT_VERSION = "2025.1"
    NG_MODE = False  # GUI 모드로 실행
    
    print(f"AEDT 버전: {AEDT_VERSION}")
    print(f"GUI 모드: {'활성화' if not NG_MODE else '비활성화'}")
    
    # 새로운 Maxwell 2D 프로젝트 생성 (GUI 모드)
    project_name_new = f"Maxwell2D_GUI_Project_{int(time.time())}"
    
    m2d_gui = Maxwell2d(
        project=project_name_new,
        design="Maxwell2D_Design",
        version=AEDT_VERSION,
        new_desktop=False,  # 기존 desktop 사용
        non_graphical=NG_MODE,  # GUI 모드
        close_on_exit=False
    )
    
    print(f"✅ Maxwell 2D GUI 프로젝트가 생성되었습니다: {project_name_new}")
    print(f"✅ 디자인명: {m2d_gui.design_name}")
    print(f"✅ 솔루션 타입: {m2d_gui.solution_type}")
    
    # GUI 창이 표시되도록 설정
    if hasattr(m2d_gui, 'odesktop'):
        try:
            # Desktop 창을 전면으로 가져오기
            m2d_gui.odesktop.RestoreWindow()
            print("✅ Maxwell 2D GUI 창이 활성화되었습니다.")
        except Exception as e:
            print(f"⚠️ 창 활성화 중 오류: {e}")
    
    print("\n🎉 Maxwell 2D GUI가 성공적으로 실행되었습니다!")
    print("💡 AEDT 창을 확인하여 Maxwell 2D 인터페이스를 사용하세요.")
    
except Exception as e:
    print(f"❌ Maxwell 2D GUI 실행 중 오류: {e}")
    print("\n🔍 문제 해결 방법:")
    print("1. Ansys AEDT가 올바르게 설치되어 있는지 확인")
    print("2. 라이선스가 사용 가능한지 확인")
    print("3. 작업 표시줄에서 AEDT 아이콘을 클릭하여 수동으로 활성화")